# Manos a la ENAHO
Demo en vivo — Charla UNI 2026

## 1. Cargar la Sumaria 2024

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_stata(
    "/Users/jose/Library/CloudStorage/Dropbox/bases/ENAHO/2024/sumaria-2024.dta",
    convert_categoricals=False
)
df.columns = df.columns.str.lower().str.strip()
print(f"Filas: {len(df):,}  |  Columnas: {len(df.columns)}")

## 2. Replicar la tasa de pobreza oficial

INEI reportó **27.6%** de pobreza monetaria en 2024. ¿Podemos replicarlo?

In [ ]:
# Gasto per cápita mensual
df["gasto_pc"] = df["gashog2d"] / (df["mieperho"] * 12)

# Clasificar pobreza
df["pobre"] = (df["gasto_pc"] < df["linea"]).astype(int)
df["pobre_extremo"] = (df["gasto_pc"] < df["linpe"]).astype(int)

# Factor de expansión poblacional
df["factor_pob"] = df["factor07"] * df["mieperho"]

# Tasa ponderada
tasa = np.average(df["pobre"], weights=df["factor_pob"]) * 100
tasa_ext = np.average(df["pobre_extremo"], weights=df["factor_pob"]) * 100

print(f"Pobreza monetaria:  {tasa:.1f}%  (oficial INEI: 27.6%)")
print(f"Pobreza extrema:    {tasa_ext:.1f}%  (oficial INEI: 5.5%)")

## 3. Perfil del pobre: urbano vs. rural

In [ ]:
df["rural"] = (df["estrato"] > 5).astype(int)

for area, nombre in [(0, "Urbano"), (1, "Rural")]:
    sub = df[df["rural"] == area]
    t = np.average(sub["pobre"], weights=sub["factor_pob"]) * 100
    print(f"{nombre}: {t:.1f}%")

## 4. Perfil del pobre: tamaño de hogar

In [ ]:
for s in [1, 2, 3, 4]:
    sub = df[df["mieperho"] == s]
    t = np.average(sub["pobre"], weights=sub["factor_pob"]) * 100
    print(f"{s} miembros:  {t:.1f}%")

sub = df[df["mieperho"] >= 5]
t = np.average(sub["pobre"], weights=sub["factor_pob"]) * 100
print(f"5+ miembros: {t:.1f}%")

## 5. Regresión: ¿qué se correlaciona con ser pobre?

**OJO:** Esto NO es causal. Solo correlaciones.

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(df[["rural", "mieperho"]])
modelo = sm.WLS(df["pobre"], X, weights=df["factor_pob"]).fit()

print(f"N = {int(modelo.nobs):,}   R² = {modelo.rsquared:.3f}\n")
print("Variable       Coef     p-value")
print("-" * 35)
for v in ["rural", "mieperho"]:
    print(f"{v:12s}  {modelo.params[v]:+.4f}   {modelo.pvalues[v]:.4f}")

print("\nInterpretación:")
print(f"  Vivir en zona rural → {modelo.params['rural']*100:+.1f} pp de probabilidad de ser pobre")
print(f"  Cada miembro adicional → {modelo.params['mieperho']*100:+.1f} pp")
print("\n⚠️  Esto es CORRELACIÓN, no causalidad.")

## 6. Desigualdad: Gini y percentiles

In [ ]:
from scipy.integrate import trapezoid
from statsmodels.stats.weightstats import DescrStatsW

# Gini ponderado
m = (df["gasto_pc"] > 0) & (df["factor_pob"] > 0)
v = df.loc[m, "gasto_pc"].values
w = df.loc[m, "factor_pob"].values
orden = np.argsort(v)
v, w = v[orden], w[orden]
gini = 1 - 2 * trapezoid(np.cumsum(v*w)/np.sum(v*w), np.cumsum(w)/np.sum(w))

print(f"Gini del gasto per cápita: {gini:.3f}")

# Percentiles
stats = DescrStatsW(df.loc[m, "gasto_pc"], weights=df.loc[m, "factor_pob"])
p10, p50, p90 = [stats.quantile(q).iloc[0] for q in [0.1, 0.5, 0.9]]

print(f"\nP10: S/ {p10:.0f}   (los más pobres gastan esto al mes)")
print(f"P50: S/ {p50:.0f}   (la mediana)")
print(f"P90: S/ {p90:.0f}  (el top 10%)")
print(f"\nRatio P90/P10: {p90/p10:.1f}x")
print("El 10% más rico gasta casi 5 veces más que el 10% más pobre.")

## Resumen

Con **Sumaria + factor07 + Python** acabamos de:

1. Replicar la tasa oficial de pobreza
2. Perfilar al pobre (rural, hogares grandes)
3. Correr regresiones (correlaciones, NO causales)
4. Calcular desigualdad (Gini, percentiles)

**Pregunta para el Bloque 2:** ¿Ser rural *causa* pobreza, o los pobres viven en zonas rurales?